# 03f | SQL interview questions

This page covers the five most common business questions in data science interviews. The question types are very stable: master these patterns and most SQL interview questions turn out to be variations of them.

About 70 minutes. Same as 02f: think it through yourself first, look at the hint if you get stuck, and redo the problems a few days later.

## How to approach a problem

1. **What does one row of the result represent?** Which columns does it have?
2. **Which table do you start from?** Make sure no unit in the "denominator" is lost.
3. **Break it into CTEs.** Be clear about what one row of each CTE represents.
4. **Call out edge cases:** duplicate records, NULLs, ties, dates with no data.

## Date functions: every database writes them differently

This is where SQL dialects differ the most. The practice environment is SQLite; in an interview any dialect is fine as long as you say "I'm using MySQL / PostgreSQL syntax". Interviewers care about the approach and will not mark you down for a function name.

| Task | SQLite (used here) | MySQL | PostgreSQL |
|---|---|---|---|
| Days between | `julianday(b) - julianday(a)` | `DATEDIFF(b, a)` | `b - a` |
| Add 1 day | `date(d, '+1 day')` | `DATE_ADD(d, INTERVAL 1 DAY)` | `d + INTERVAL '1 day'` |
| Year-month | `strftime('%Y-%m', d)` | `DATE_FORMAT(d, '%Y-%m')` | `TO_CHAR(d, 'YYYY-MM')` |
| Truncate to month start | `date(d, 'start of month')` | `DATE_FORMAT(d, '%Y-%m-01')` | `DATE_TRUNC('month', d)` |

In [ ]:
SELECT order_id, order_date,
       strftime('%Y-%m', order_date)                     AS month,
       date(order_date, '+7 day')                        AS plus_7_days,
       julianday(order_date) - julianday('2026-01-01')   AS days_since_jan1
FROM orders
LIMIT 4;

### ✏️ Exercise 1 (write it yourself)

**Problem 1: Monthly sales**

Compute the total amount of completed orders per month. Output `month` (like `'2026-01'`) and `total`, sorted by month ascending.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Answer</b></summary>

```sql
SELECT strftime('%Y-%m', order_date) AS month,
       SUM(amount)                   AS total
FROM orders
WHERE status = 'completed'
GROUP BY month
ORDER BY month;
```

</details>

### ✏️ Exercise 2 (write it yourself)

**Problem 2: Daily active users (DAU)**

Using the records in `events` with `event_type = 'login'`, compute the number of **distinct users** who logged in each day. Output `event_date`, `dau`, sorted by date ascending.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

The key is `COUNT(DISTINCT user_id)`. User 1 has two login records on January 2, so `COUNT(*)` would overcount.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT event_date, COUNT(DISTINCT user_id) AS dau
FROM events
WHERE event_type = 'login'
GROUP BY event_date
ORDER BY event_date;
```

</details>

## Problem 3: Day-1 retention

Definition: treat each user's **first login** date as day 0; if they also log in on day 1 (the next day), they count as retained on day 1.

The pattern is the same as in 02f, written as three steps:

1. `logins`: deduplicated login records (one row per user per day);
2. `first`: each user's first login date;
3. Left join `logins` onto `first` on "same user, and date equals the first date plus 1 day". Users who match are the retained ones.

Because the left join starts from `first`, every user is in the denominator.

### ✏️ Exercise 3 (fill in the blanks)

Complete the day-1 retention query, outputting a single column `retention`.

In [ ]:
WITH logins AS (
  SELECT DISTINCT user_id, event_date
  FROM events
  WHERE event_type = 'login'
),
first AS (
  SELECT user_id, ____(event_date) AS first_date
  FROM logins
  GROUP BY user_id
)
SELECT 1.0 * COUNT(l.user_id) / COUNT(*) AS retention
FROM first f
LEFT JOIN logins l
  ON l.user_id = f.user_id
 AND l.event_date = date(f.first_date, ____);

<details>
<summary><b>Hint</b></summary>

The first blank is `MIN`; the second is `'+1 day'` (with the single quotes).

</details>

<details>
<summary><b>Answer</b></summary>

```sql
WITH logins AS (
  SELECT DISTINCT user_id, event_date
  FROM events
  WHERE event_type = 'login'
),
first AS (
  SELECT user_id, MIN(event_date) AS first_date
  FROM logins
  GROUP BY user_id
)
SELECT 1.0 * COUNT(l.user_id) / COUNT(*) AS retention
FROM first f
LEFT JOIN logins l
  ON l.user_id = f.user_id
 AND l.event_date = date(f.first_date, '+1 day');
```

</details>

> **Why does `COUNT(*)` equal the number of users?** `first` has one row per user, and `logins` is already deduplicated, so each user matches at most one row on "first date plus 1 day". After the left join there is still one row per user. `COUNT(l.user_id)` counts only the matches, i.e. the retained users.
>
> If you forget to deduplicate `logins`, a user who logged in twice on the same day is counted twice. **Deduplicate first** is step one of any retention problem.

### ✏️ Exercise 4 (write it yourself)

**Problem 3 variant: day-1 retention by first-login date (cohort)**

For each first-login date (`first_date`), compute the number of new users that day, `n_users`, and how many of them were retained on day 1, `n_retained`. Output `first_date`, `n_users`, `n_retained`, sorted by date ascending.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

Almost the same as the previous exercise; just add `GROUP BY f.first_date` at the end and output `COUNT(*)` and `COUNT(l.user_id)` separately.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
WITH logins AS (
  SELECT DISTINCT user_id, event_date
  FROM events
  WHERE event_type = 'login'
),
first AS (
  SELECT user_id, MIN(event_date) AS first_date
  FROM logins
  GROUP BY user_id
)
SELECT f.first_date,
       COUNT(*)         AS n_users,
       COUNT(l.user_id) AS n_retained
FROM first f
LEFT JOIN logins l
  ON l.user_id = f.user_id
 AND l.event_date = date(f.first_date, '+1 day')
GROUP BY f.first_date
ORDER BY f.first_date;
```

</details>

## Problem 4: Funnel

The purchase path is view (`view`) → add to cart (`cart`) → purchase (`purchase`). Funnel analysis counts the **distinct users** at each step.

Conditional aggregation does it in one pass: `COUNT(DISTINCT CASE WHEN event_type = 'view' THEN user_id END)`. Without an `ELSE`, `CASE` returns NULL for rows that do not match, and `COUNT` does not count NULLs.

### ✏️ Exercise 5 (write it yourself)

Count the distinct users at each of the three funnel steps, outputting one row with three columns: `n_view`, `n_cart`, `n_purchase`.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Answer</b></summary>

```sql
SELECT COUNT(DISTINCT CASE WHEN event_type = 'view'     THEN user_id END) AS n_view,
       COUNT(DISTINCT CASE WHEN event_type = 'cart'     THEN user_id END) AS n_cart,
       COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS n_purchase
FROM events;
```

The result is 10 → 7 → 5: a 70% add-to-cart rate and about 71% conversion from cart to purchase. In an interview, once you have the numbers, volunteer the conversion rate of each step and which step loses the most users.

A strict funnel also requires ordering (view before cart) and a time window (e.g. within the same day). In an interview, clarify the definition first, then decide whether a time condition is needed.

</details>

## Problem 5: Consecutive logins (gaps and islands)

"Find users who logged in on at least 3 consecutive days" is the most famous hard SQL interview question, but the pattern is fixed.

**The core trick:** for each user, sort the login dates and number them. If the dates are consecutive, "date − row number" is a **constant**; as soon as there is a break, the value changes.

Take user 1 as an example:

| Login date | Row number rn | Date − rn days |
|---|---|---|
| 01-01 | 1 | 12-31 |
| 01-02 | 2 | 12-31 |
| 01-03 | 3 | 12-31 |
| 01-05 | 4 | 01-01 |
| 01-08 | 5 | 01-03 |

The first three rows share the same "date − rn", so they form one consecutive run. Hence: group by (user, date − rn); each group is one streak of logins, and the number of rows in the group is the streak length.

In [ ]:
WITH logins AS (
  SELECT DISTINCT user_id, event_date
  FROM events
  WHERE event_type = 'login'
),
numbered AS (
  SELECT user_id, event_date,
         ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_date) AS rn
  FROM logins
)
SELECT user_id, event_date, rn,
       date(event_date, '-' || rn || ' day') AS grp
FROM numbered
WHERE user_id = 1;

`date(event_date, '-' || rn || ' day')` is how SQLite writes "date minus rn days" (`||` concatenates strings). In MySQL it is `DATE_SUB(event_date, INTERVAL rn DAY)`; in PostgreSQL, `event_date - rn`.

### ✏️ Exercise 6 (write it yourself)

Find users who **ever logged in on at least 3 consecutive days**. Output a single column `user_id` (each user once).

In [ ]:
WITH logins AS (
  SELECT DISTINCT user_id, event_date
  FROM events
  WHERE event_type = 'login'
),
numbered AS (
  SELECT user_id, event_date,
         ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_date) AS rn
  FROM logins
),
grouped AS (
  SELECT user_id, event_date,
         date(event_date, '-' || rn || ' day') AS grp
  FROM numbered
)
-- Continue below: group by user_id and grp, and keep groups with >= 3 rows
SELECT user_id, grp
FROM grouped;

<details>
<summary><b>Hint</b></summary>

Change the last part to: `SELECT DISTINCT user_id FROM grouped GROUP BY user_id, grp HAVING COUNT(*) >= 3`.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
WITH logins AS (
  SELECT DISTINCT user_id, event_date
  FROM events
  WHERE event_type = 'login'
),
numbered AS (
  SELECT user_id, event_date,
         ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_date) AS rn
  FROM logins
),
grouped AS (
  SELECT user_id, event_date,
         date(event_date, '-' || rn || ' day') AS grp
  FROM numbered
)
SELECT DISTINCT user_id
FROM grouped
GROUP BY user_id, grp
HAVING COUNT(*) >= 3;
```

</details>

## Month-over-month growth (example)

"How much did sales grow each month compared with the previous month": aggregate by month first, then use `LAG` to get the previous month.

In [ ]:
WITH monthly AS (
  SELECT strftime('%Y-%m', order_date) AS month, SUM(amount) AS total
  FROM orders
  WHERE status = 'completed'
  GROUP BY month
)
SELECT month, total,
       LAG(total) OVER (ORDER BY month)                                   AS prev_total,
       ROUND(1.0 * total / LAG(total) OVER (ORDER BY month) - 1, 3)       AS growth
FROM monthly;

**🗣 Say it out loud:** Which behaviors of NULL in SQL do you need to watch out for?

<details>
<summary><b>Reference answer</b></summary>

① No comparison with NULL is ever true, so test for missing values with `IS NULL`; ② `WHERE x <> 'a'` does not return rows where x is NULL; ③ aggregate functions (`SUM`, `AVG`, `COUNT(col)`) skip NULLs, but `COUNT(*)` does not; ④ with `NOT IN (subquery)`, if the subquery returns any NULL the whole condition is never true, so `NOT EXISTS` or a left join is safer; ⑤ in joins, NULL does not equal NULL, so NULL keys never match; ⑥ use `COALESCE(x, default)` to handle missing values.

**Short version:** Comparisons with NULL are never true, so I use IS NULL. Aggregates skip NULLs except COUNT(*). NOT IN with a NULL in the list returns nothing, so I prefer NOT EXISTS or an anti-join. NULL keys don't match in joins, and COALESCE supplies defaults.

</details>

**🗣 Say it out loud:** How would you compute "7-day rolling active users" (for each day, the number of distinct users who logged in during the 7 days up to that day)?

<details>
<summary><b>Reference answer</b></summary>

You cannot simply use a window frame like `ROWS BETWEEN 6 PRECEDING`, because you are counting **distinct users**: window functions do not support `COUNT(DISTINCT)`, and there may be gaps in the dates. The robust approach is a self-join: build a calendar table `days` (one row per day) and a deduplicated login table `logins`, then `days d JOIN logins l ON l.event_date BETWEEN d.day - 6 days AND d.day`, and `GROUP BY d.day` with `COUNT(DISTINCT l.user_id)`. For an additive metric such as "7-day rolling sales", aggregate by day first and then use `SUM(...) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)`, stating the assumption that every day has a row.

**Short version:** For a rolling distinct count I'd self-join a calendar of days to deduplicated logins on a 7-day date range and count distinct users per day, because window functions can't do COUNT DISTINCT. For an additive metric I'd aggregate by day first and use a window frame of 6 preceding rows.

</details>

## SQL checkpoint

If you can write at least 5 of the following on your own without looking at the answers, you have passed the SQL checkpoint:

1. `GROUP BY` + `HAVING` (03b);
2. Conditional aggregation `SUM(CASE WHEN ...)` (03b);
3. A left join that records missing values as 0, with filters in the right place (03c);
4. Step-by-step queries with CTEs (03d);
5. Top N per group (03e);
6. Running totals and `LAG` (03e);
7. Day-1 retention (this page);
8. Explaining the trick for consecutive-login questions (this page).

> The biggest risk with SQL is getting rusty. Come back every week and redo two or three of the "write it yourself" exercises until your interviews are over.